In [1]:
# ===== STEP 7.2a: Install the GGUF "player" =====
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

In [2]:
# ===== STEP 7.2b: Download the GGUF, load it on CPU, ask a question =====
import os, time
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

REPO_GGUF = "pramod586/mistral-7b-customer-support-GGUF"
FILE_GGUF = "mistral-7b-customer-support-Q4_K_M.gguf"

# 1) Download (about 4.4 GB)
t0 = time.time()
path = hf_hub_download(repo_id=REPO_GGUF, filename=FILE_GGUF)
print(f"Downloaded {os.path.getsize(path)/1e9:.2f} GB in {(time.time()-t0)/60:.1f} min")

# 2) Load it on the CPU
t0 = time.time()
llm = Llama(model_path=path, n_ctx=2048, n_threads=os.cpu_count(), verbose=False)
print(f"Loaded in {time.time()-t0:.0f} s using {os.cpu_count()} CPU cores")

# 3) Ask the same question as Steps 2.3 and 4.6
question = "I want to cancel my order, how do I do it?"
t0 = time.time()
out = llm.create_chat_completion(
    messages=[{"role": "user", "content": question}],
    max_tokens=400,
    temperature=0,          # no randomness, same as our exam
)
secs   = time.time() - t0
answer = out["choices"][0]["message"]["content"].strip()
n_tok  = out["usage"]["completion_tokens"]

print("\n===== STEP 7.2 CHECK =====")
print("CUSTOMER:", question)
print("\nGGUF MODEL (CPU):\n", answer)
print(f"\nAnswer length   : {len(answer.split())} words / {n_tok} tokens")
print("Stopped on its own:", out["choices"][0]["finish_reason"] == "stop")
print(f"Time            : {secs:.0f} s  (~{n_tok/secs:.1f} tokens per second)")

mistral-7b-customer-support-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 4.37GB            

mistral-7b-customer-support-Q4_K_M.gguf: downloading bytes:           |  0.00B            

Downloaded 4.37 GB in 0.8 min
Loaded in 20 s using 2 CPU cores

===== STEP 7.2 CHECK =====
CUSTOMER: I want to cancel my order, how do I do it?

GGUF MODEL (CPU):
 I'm sorry to hear that you want to cancel your order. To assist you with the cancellation process, please provide me with the order number or any other relevant details. Once I have the necessary information, I will guide you through the steps to cancel your order successfully.

Answer length   : 48 words / 55 tokens
Stopped on its own: True
Time            : 37 s  (~1.5 tokens per second)
